# Zero-Day Detection, Isolation Forest, and Hybrid IDS

This notebook is split from `majorproject(1).ipynb`. Run cells from top to bottom.

In [21]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os 

In [22]:
folder_path = r"C:\Users\Asus\Downloads\OneDrive_1_8-22-2026"
csv_files = glob.glob(os.path.join(folder_path,"*.csv"))
dataframes =[]
for file in csv_files:
    temp_df = pd.read_csv(file)
    dataframes.append(temp_df)

In [23]:
df =  pd.concat(dataframes,ignore_index=True)
df.columns = df.columns.str.strip()
df = df.drop_duplicates()
print(df.iloc[:, -1].value_counts())

label
1    164673
0     93000
Name: count, dtype: int64


In [24]:
y = df["label"]
X = df.drop(columns=["label", "attack_cat", "id"])
X = pd.get_dummies(X, drop_first=True)
X = X.astype(np.float64)
X = X.to_numpy()
y = y.to_numpy()
X[X == np.inf] = np.nan
X[X == -np.inf] = np.nan

In [25]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state = 42)


In [26]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(missing_values= np.nan,strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [27]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [28]:
print(np.isnan(X_train).sum())
print(np.isnan(X_test).sum())

0
0


In [29]:
print(np.isinf(X_train).sum())
print(np.isinf(X_test).sum())

0
0


In [30]:

import numpy as np

# Preserve the existing feature matrix
X = np.asarray(X, dtype=np.float64)

print("X shape:", X.shape)
print("Number of features:", X.shape[1])


X shape: (257673, 193)
Number of features: 193


In [31]:

# Step 19: Inspect attack categories

print("Available attack categories:")
print(df["attack_cat"].value_counts())


Available attack categories:
attack_cat
Normal            93000
Generic           58871
Exploits          44525
Fuzzers           24246
DoS               16353
Reconnaissance    13987
Analysis           2677
Backdoor           2329
Shellcode          1511
Worms               174
Name: count, dtype: int64


In [32]:

import numpy as np
import pandas as pd

# Step 19: Hold out the Worms attack category
target_col = "attack_cat"

# Separate unseen attacks from known traffic
unseen_mask = df[target_col].eq("Worms")

df_known = df.loc[~unseen_mask].copy()
df_unseen = df.loc[unseen_mask].copy()

print("Known traffic records:", len(df_known))
print("Unseen Worms records:", len(df_unseen))

print("\nKnown attack categories:")
print(df_known[target_col].value_counts())

print("\nUnseen category:")
print(df_unseen[target_col].value_counts())


Known traffic records: 257499
Unseen Worms records: 174

Known attack categories:
attack_cat
Normal            93000
Generic           58871
Exploits          44525
Fuzzers           24246
DoS               16353
Reconnaissance    13987
Analysis           2677
Backdoor           2329
Shellcode          1511
Name: count, dtype: int64

Unseen category:
attack_cat
Worms    174
Name: count, dtype: int64


In [33]:

# Check which preprocessing and model variables exist

check_vars = [
    "X", "y", "X_train", "X_test",
    "y_train", "y_test", "best_rf",
    "rf_model", "sc", "imputer",
    "le", "feature_names"
]

for var in check_vars:
    if var in globals():
        obj = globals()[var]
        print(f"{var}: {type(obj).__name__}", 
              getattr(obj, "shape", ""))
    else:
        print(f"{var}: Not found")


X: ndarray (257673, 193)
y: ndarray (257673,)
X_train: ndarray (206138, 193)
X_test: ndarray (51535, 193)
y_train: ndarray (206138,)
y_test: ndarray (51535,)
best_rf: Not found
rf_model: Not found
sc: StandardScaler 
imputer: SimpleImputer 
le: Not found
feature_names: Not found


In [34]:

# Step 19.2: Train a model without Worms attacks

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

# Separate input features and binary target
# Exclude attack_cat to prevent target leakage
X_known_df = df_known.drop(
    columns=["label", "attack_cat", "id"],
    errors="ignore"
)
y_known = df_known["label"]

X_worms_df = df_unseen.drop(
    columns=["label", "attack_cat", "id"],
    errors="ignore"
)
y_worms = df_unseen["label"]

# Encode categorical features consistently
X_known_df = pd.get_dummies(X_known_df, drop_first=True)
X_worms_df = pd.get_dummies(X_worms_df, drop_first=True)

# Match the training feature columns
X_worms_df = X_worms_df.reindex(
    columns=X_known_df.columns,
    fill_value=0
)

# Convert invalid values to missing values
X_known_df = X_known_df.replace([np.inf, -np.inf], np.nan)
X_worms_df = X_worms_df.replace([np.inf, -np.inf], np.nan)

# Split known traffic for training and validation
X_train19, X_val19, y_train19, y_val19 = train_test_split(
    X_known_df,
    y_known,
    test_size=0.2,
    random_state=42,
    stratify=y_known
)

# Fit preprocessing on training data only
imputer19 = SimpleImputer(strategy="mean")
X_train19 = imputer19.fit_transform(X_train19)
X_val19 = imputer19.transform(X_val19)
X_worms = imputer19.transform(X_worms_df)

scaler19 = StandardScaler()
X_train19 = scaler19.fit_transform(X_train19)
X_val19 = scaler19.transform(X_val19)
X_worms = scaler19.transform(X_worms)

# Train a binary intrusion detector without Worms
zero_day_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

zero_day_model.fit(X_train19, y_train19)

print("Training completed without Worms records.")
print("Training records:", len(y_train19))
print("Validation records:", len(y_val19))
print("Held-out Worms records:", len(y_worms))
print("Model classes:", zero_day_model.classes_)


Training completed without Worms records.
Training records: 205999
Validation records: 51500
Held-out Worms records: 174
Model classes: [0 1]


In [35]:
from pathlib import Path
import joblib

# Create the model directory
MODEL_DIR = Path("final_ids_models")
MODEL_DIR.mkdir(exist_ok=True)

# Save the trained model
joblib.dump(
    zero_day_model,
    MODEL_DIR / "random_forest_zero_day.pkl"
)

# Save the exact feature names and order
joblib.dump(
    X_known_df.columns.tolist(),
    MODEL_DIR / "feature_names.pkl"
)

# Save the preprocessing objects fitted during training
joblib.dump(
    imputer19,
    MODEL_DIR / "imputer.pkl"
)

joblib.dump(
    scaler19,
    MODEL_DIR / "scaler.pkl"
)

print("Model and preprocessing artifacts saved successfully.")
print("Number of features:", len(X_known_df.columns))

Model and preprocessing artifacts saved successfully.
Number of features: 193


In [36]:

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# Predict on the 174 unseen Worms records
worms_predictions = zero_day_model.predict(X_worms)

print("Actual Worms labels:")
print(pd.Series(y_worms).value_counts())

print("\nPredictions for Worms traffic:")
print(pd.Series(worms_predictions).value_counts())

# Check how many Worms attacks were detected
attack_label = 1

detected = np.sum(worms_predictions == attack_label)
total = len(worms_predictions)

print(f"\nWorms flagged as attacks: {detected}/{total}")
print(f"Detection rate: {detected / total:.2%}")

# Evaluate performance on known traffic
val_predictions = zero_day_model.predict(X_val19)

print("\nValidation Classification Report:")
print(classification_report(
    y_val19,
    val_predictions,
    zero_division=0
))

print("\nValidation Confusion Matrix:")
print(confusion_matrix(y_val19, val_predictions))


Actual Worms labels:
label
1    174
Name: count, dtype: int64

Predictions for Worms traffic:
1    174
Name: count, dtype: int64

Worms flagged as attacks: 174/174
Detection rate: 100.00%

Validation Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.96      0.92     18600
           1       0.98      0.93      0.95     32900

    accuracy                           0.94     51500
   macro avg       0.93      0.95      0.94     51500
weighted avg       0.94      0.94      0.94     51500


Validation Confusion Matrix:
[[17912   688]
 [ 2298 30602]]


In [37]:

# Step 20: Unsupervised anomaly detection using Isolation Forest

from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Use known training traffic only
# X_train19 has already been imputed and scaled
isolation_model = IsolationForest(
    n_estimators=100,
    contamination=0.10,
    random_state=42,
    n_jobs=-1
)

isolation_model.fit(X_train19)

# Evaluate on known validation traffic
# Isolation Forest returns 1 for normal and -1 for anomalies
iso_val_raw = isolation_model.predict(X_val19)

# Convert predictions to the project's label convention:
# 0 = normal, 1 = attack/anomaly
iso_val_predictions = np.where(iso_val_raw == -1, 1, 0)

print("Isolation Forest — Known Traffic Validation")
print(classification_report(
    y_val19,
    iso_val_predictions,
    target_names=["Normal", "Attack"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_val19, iso_val_predictions))

# Test on the held-out Worms records
iso_worms_raw = isolation_model.predict(X_worms)
iso_worms_predictions = np.where(iso_worms_raw == -1, 1, 0)

worms_detected_iso = np.sum(iso_worms_predictions == 1)

print("\nIsolation Forest — Unseen Worms")
print(f"Detected as anomalies: {worms_detected_iso}/{len(X_worms)}")
print(f"Anomaly detection rate: {worms_detected_iso / len(X_worms):.2%}")
abs

Isolation Forest — Known Traffic Validation
              precision    recall  f1-score   support

      Normal       0.35      0.87      0.50     18600
      Attack       0.53      0.08      0.14     32900

    accuracy                           0.37     51500
   macro avg       0.44      0.48      0.32     51500
weighted avg       0.47      0.37      0.27     51500

Confusion Matrix:
[[16204  2396]
 [30153  2747]]

Isolation Forest — Unseen Worms
Detected as anomalies: 32/174
Anomaly detection rate: 18.39%


<function abs(x, /)>

In [38]:

# Step 21: Hybrid Intrusion Detection System

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Random Forest predictions on known validation traffic
rf_val_predictions = zero_day_model.predict(X_val19)

# Isolation Forest predictions: -1 = anomaly, 1 = normal
iso_val_raw = isolation_model.predict(X_val19)

# Convert to binary labels: 0 = normal, 1 = attack
iso_val_predictions = np.where(iso_val_raw == -1, 1, 0)

# Hybrid rule: flag traffic if either model predicts an attack
hybrid_val_predictions = np.where(
    (rf_val_predictions == 1) |
    (iso_val_predictions == 1),
    1,
    0
)

print("Hybrid IDS — Known Traffic Validation")
print(classification_report(
    y_val19,
    hybrid_val_predictions,
    target_names=["Normal", "Attack"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_val19, hybrid_val_predictions))

# Evaluate on held-out Worms attacks
rf_worms_predictions = zero_day_model.predict(X_worms)
iso_worms_raw = isolation_model.predict(X_worms)
iso_worms_predictions = np.where(iso_worms_raw == -1, 1, 0)

hybrid_worms_predictions = np.where(
    (rf_worms_predictions == 1) |
    (iso_worms_predictions == 1),
    1,
    0
)

detected = np.sum(hybrid_worms_predictions == 1)

print("\nHybrid IDS — Unseen Worms")
print(f"Detected: {detected}/{len(X_worms)}")
print(f"Detection rate: {detected / len(X_worms):.2%}")


Hybrid IDS — Known Traffic Validation
              precision    recall  f1-score   support

      Normal       0.88      0.84      0.86     18600
      Attack       0.91      0.93      0.92     32900

    accuracy                           0.90     51500
   macro avg       0.89      0.88      0.89     51500
weighted avg       0.90      0.90      0.90     51500

Confusion Matrix:
[[15572  3028]
 [ 2212 30688]]

Hybrid IDS — Unseen Worms
Detected: 174/174
Detection rate: 100.00%


In [39]:

import joblib
import json
import os

# Create a folder for final project artifacts
os.makedirs("final_ids_models", exist_ok=True)

# Save the trained models
joblib.dump(
    zero_day_model,
    "final_ids_models/random_forest_zero_day.pkl"
)

joblib.dump(
    isolation_model,
    "final_ids_models/isolation_forest.pkl"
)

# Save preprocessing objects
joblib.dump(
    imputer19,
    "final_ids_models/imputer.pkl"
)

joblib.dump(
    scaler19,
    "final_ids_models/scaler.pkl"
)

# Save the exact feature order used for training
joblib.dump(
    X_known_df.columns.tolist(),
    "final_ids_models/feature_names.pkl"
)

# Save project metadata
metadata = {
    "project": "Network Intrusion Detection System",
    "supervised_model": "Random Forest",
    "anomaly_model": "Isolation Forest",
    "hybrid_rule": "Flag as attack if either model predicts attack",
    "attack_label": 1,
    "normal_label": 0,
    "held_out_attack_category": "Worms"
}

with open("final_ids_models/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("All model artifacts saved successfully.")
print("Folder: final_ids_models")
print(os.listdir("final_ids_models"))


All model artifacts saved successfully.
Folder: final_ids_models
['best_ids_model.pkl', 'feature_names.pkl', 'imputer.pkl', 'isolation_forest.pkl', 'isolation_forest_model.pkl', 'metadata.json', 'model_comparison.csv', 'preprocessor.pkl', 'random_forest_zero_day.pkl', 'scaler.pkl', 'xgb_fraud_model.pkl']


In [40]:

# Step 23: End-to-end model verification

import os
import joblib
import numpy as np
import pandas as pd

# Check saved model files
model_dir = "final_ids_models"

required_files = [
    "random_forest_zero_day.pkl",
    "isolation_forest.pkl",
    "imputer.pkl",
    "scaler.pkl",
    "feature_names.pkl",
    "metadata.json"
]

missing = [
    filename for filename in required_files
    if not os.path.exists(os.path.join(model_dir, filename))
]

if missing:
    print("Missing files:", missing)
    print("Run the model-saving cell first.")
else:
    print("All saved artifacts found.")

    # Load saved artifacts
    rf_loaded = joblib.load(
        os.path.join(model_dir, "random_forest_zero_day.pkl")
    )
    iso_loaded = joblib.load(
        os.path.join(model_dir, "isolation_forest.pkl")
    )
    imputer_loaded = joblib.load(
        os.path.join(model_dir, "imputer.pkl")
    )
    scaler_loaded = joblib.load(
        os.path.join(model_dir, "scaler.pkl")
    )
    saved_features = joblib.load(
        os.path.join(model_dir, "feature_names.pkl")
    )

    # Verify the saved models on existing preprocessed samples
    X_sample = X_val19[:10]

    rf_predictions = rf_loaded.predict(X_sample)
    iso_predictions = iso_loaded.predict(X_sample)

    # Isolation Forest: -1 = anomaly, 1 = normal
    iso_binary = np.where(iso_predictions == -1, 1, 0)

    hybrid_predictions = np.where(
        (rf_predictions == 1) | (iso_binary == 1),
        1,
        0
    )

    print("\nEnd-to-end test successful.")
    print("Samples tested:", len(X_sample))
    print("Random Forest predictions:", rf_predictions)
    print("Isolation Forest predictions:", iso_binary)
    print("Hybrid predictions:", hybrid_predictions)

    print("\nFeature count:", len(saved_features))
    print("Expected feature count:", X_sample.shape[1])

    assert len(saved_features) == X_sample.shape[1], (
        "Feature count mismatch!"
    )

    print("\nStep 23 completed successfully.")


All saved artifacts found.

End-to-end test successful.
Samples tested: 10
Random Forest predictions: [1 0 1 1 1 1 1 0 1 1]
Isolation Forest predictions: [0 0 0 0 0 0 0 0 0 1]
Hybrid predictions: [1 0 1 1 1 1 1 0 1 1]

Feature count: 193
Expected feature count: 193

Step 23 completed successfully.


In [41]:

# Step 24: Final project results summary

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Evaluate the hybrid IDS on known validation traffic
rf_val = zero_day_model.predict(X_val19)

iso_val = isolation_model.predict(X_val19)
iso_val = (iso_val == -1).astype(int)

hybrid_val = ((rf_val == 1) | (iso_val == 1)).astype(int)

results = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Accuracy": accuracy_score(y_val19, rf_val),
        "Attack Precision": precision_score(y_val19, rf_val, zero_division=0),
        "Attack Recall": recall_score(y_val19, rf_val, zero_division=0),
        "Attack F1": f1_score(y_val19, rf_val, zero_division=0)
    },
    {
        "Model": "Isolation Forest",
        "Accuracy": accuracy_score(y_val19, iso_val),
        "Attack Precision": precision_score(y_val19, iso_val, zero_division=0),
        "Attack Recall": recall_score(y_val19, iso_val, zero_division=0),
        "Attack F1": f1_score(y_val19, iso_val, zero_division=0)
    },
    {
        "Model": "Hybrid IDS",
        "Accuracy": accuracy_score(y_val19, hybrid_val),
        "Attack Precision": precision_score(y_val19, hybrid_val, zero_division=0),
        "Attack Recall": recall_score(y_val19, hybrid_val, zero_division=0),
        "Attack F1": f1_score(y_val19, hybrid_val, zero_division=0)
    }
])

print("FINAL MODEL COMPARISON")
print(results.round(4).to_string(index=False))

results.to_csv("final_ids_models/model_comparison.csv", index=False)

print("\nResults saved to final_ids_models/model_comparison.csv")


FINAL MODEL COMPARISON
           Model  Accuracy  Attack Precision  Attack Recall  Attack F1
   Random Forest    0.9420            0.9780         0.9302     0.9535
Isolation Forest    0.3680            0.5341         0.0835     0.1444
      Hybrid IDS    0.8983            0.9102         0.9328     0.9213

Results saved to final_ids_models/model_comparison.csv
